# 03 — Evaluation: Base vs. SFT vs. DPO
### Comparing all three checkpoints on the same fixed question set

This notebook loads the **base model**, the **SFT adapter**, and the **DPO adapter** and runs each through `evaluation_questions.json`, producing a side-by-side comparison table - the evidence behind the `## Results` section of the project README.


## Step 0 — Install dependencies + imports

In [ ]:
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U trl peft accelerate bitsandbytes transformers datasets pandas


In [ ]:
import json
import torch
import pandas as pd
from unsloth import FastLanguageModel


## Step 1 — Load the base model, then the SFT and DPO checkpoints

**What:** load the frozen base model once, and generate from it three ways:
1. with **no adapter** (pure base model),
2. with the **SFT adapter** attached,
3. with the **DPO adapter** attached.

**Why load once, swap adapters:** this is the memory-efficient way to compare all three states - LoRA adapters are small, so swapping between them is far cheaper than loading three full copies of the model into Colab's limited VRAM.

**How:** load base weights, then use `model.load_adapter(...)` / `model.set_adapter(...)` to switch which adapter is active before each round of generation.

In [ ]:
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
SFT_ADAPTER_DIR = "outputs/sft_adapter"
DPO_ADAPTER_DIR = "outputs/dpo_adapter"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# WHAT: attach both adapters up front, under distinct names, so we can switch
#       between them (and a "no adapter" base state) without reloading the model.
model.load_adapter(SFT_ADAPTER_DIR, adapter_name="sft")
model.load_adapter(DPO_ADAPTER_DIR, adapter_name="dpo")

FastLanguageModel.for_inference(model)


## Step 2 — Load the evaluation questions

**What:** load the fixed, held-out question set.
**Why fixed and held-out:** using the *same* questions across all three checkpoints (and not questions seen during SFT/DPO training) is what makes the comparison fair rather than anecdotal.

In [ ]:
with open("data/evaluation_questions.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# WHAT: normalize to a plain list of question strings, however the JSON is shaped
#       (a list of strings, or a list of {"question": ...} objects).
if isinstance(eval_data[0], dict):
    questions = [item["question"] for item in eval_data]
else:
    questions = eval_data

print(f"Loaded {len(questions)} evaluation questions")


## Step 3 — Generation helper

**What:** one function that generates an answer given a question and which adapter (if any) is active.
**Why a shared function:** guarantees identical decoding settings (temperature, max tokens, system prompt) across all three models - otherwise differences in output could come from decoding, not training.

In [ ]:
SYSTEM_PROMPT = "You are PostTraining Tutor, an assistant that explains LLM training concepts (Transformers, LoRA, QLoRA, SFT, DPO, RLHF, alignment) clearly and concisely."

def generate_answer(question, adapter_name=None):
    # WHAT: point the model at a specific adapter, or disable adapters entirely for the base model.
    if adapter_name is None:
        model.disable_adapters()
    else:
        model.enable_adapters()
        model.set_adapter(adapter_name)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    # WHY do_sample=False here (unlike Notebooks 1/2): evaluation should be
    # deterministic/greedy so re-running this notebook reproduces the same comparison table.
    output = model.generate(input_ids=inputs, max_new_tokens=200, do_sample=False)
    return tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)


## Step 4 — Run all three models over every question

**What:** loop over the evaluation set, generating a Base, SFT, and DPO answer for each question.
**Why loop rather than batch:** keeps the code simple and readable for a demo/talk; batching would be a natural optimization for a larger question set.

In [ ]:
results = []

for q in questions:
    base_answer = generate_answer(q, adapter_name=None)
    sft_answer = generate_answer(q, adapter_name="sft")
    dpo_answer = generate_answer(q, adapter_name="dpo")
    results.append({
        "question": q,
        "base_model": base_answer,
        "sft_model": sft_answer,
        "dpo_model": dpo_answer,
    })
    print(f"Done: {q[:60]}...")


## Step 5 — Build and save the comparison table

**What:** turn the results into a pandas DataFrame, display it, and save it to `outputs/evaluation_results.csv`.
**Why CSV:** easy to paste straight into the README's Results table or a presentation slide.

In [ ]:
df = pd.DataFrame(results)
pd.set_option("display.max_colwidth", 120)
display(df)

df.to_csv("outputs/evaluation_results.csv", index=False)
print("Saved outputs/evaluation_results.csv")


## Step 6 — (Optional) simple quantitative signal: response length

**What:** a lightweight, non-judgmental numeric signal to accompany the qualitative reading - average response length per model.
**Why include it:** a single cheap metric (e.g., DPO answers trending shorter/more direct than base) gives the audience something concrete to look at before diving into the qualitative examples. This is **not** a claim of "better" by itself - pair it with a manual read of a few examples in `reports/results_analysis.md`.

In [ ]:
df["base_len"] = df["base_model"].str.split().apply(len)
df["sft_len"] = df["sft_model"].str.split().apply(len)
df["dpo_len"] = df["dpo_model"].str.split().apply(len)

summary = df[["base_len", "sft_len", "dpo_len"]].mean().rename("avg_words")
print(summary)


**This is the evidence layer for the project.** Paste a few rows of `outputs/evaluation_results.csv` into the README's `## 8. Results` section, and use `reports/results_analysis.md` (Response 3) to write up the qualitative takeaways for the talk.